In [1]:
!pip install transformers datasets accelerate evaluate -q

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

In [2]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
import torch

In [3]:
stories = [
    "Once upon a time in a silent forest, a glowing bird led a lost child to a hidden village made of crystal.",
    "On the edge of the moon, a lonely robot planted golden flowers that bloomed only when someone remembered home.",
    "A girl found an old key beneath her pillow and discovered it could unlock doors inside her dreams.",
    "Every night, the stars whispered secrets to a fisherman who sailed across a sky-colored sea.",
    "In a city where shadows were sold in markets, a boy bought one that could speak.",
    "A dragon who had forgotten how to breathe fire learned to sing instead, and his songs changed the weather.",
    "Deep under the ocean, a library of glowing shells kept the memories of vanished kingdoms.",
    "A clockmaker built a tiny mechanical sparrow that could pause time for exactly ten seconds.",
    "In the middle of the desert stood a blue door that appeared only during sandstorms.",
    "A young painter discovered that anything she painted under moonlight came alive before sunrise."
]

dataset = Dataset.from_dict({"text": stories})
dataset

Dataset({
    features: ['text'],
    num_rows: 10
})

In [4]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
model.config.pad_token_id = tokenizer.pad_token_id

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_dataset

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 10
})

In [6]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [12]:
training_args = TrainingArguments(
    output_dir="./gpt2-story-model",
    num_train_epochs=20,
    per_device_train_batch_size=2,
    save_steps=50,
    save_total_limit=1,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.921077
20,2.314595
30,1.328600
40,0.692599
50,0.331982
60,0.241913
70,0.223688
80,0.217469


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,3.921077
20,2.314595
30,1.328600
40,0.692599
50,0.331982
60,0.241913
70,0.223688
80,0.217469
90,0.188056


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
trainer.save_model("./gpt2-story-model")
tokenizer.save_pretrained("./gpt2-story-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-story-model/tokenizer_config.json',
 './gpt2-story-model/tokenizer.json')

In [15]:
prompt = "Once upon a time in a forgotten kingdom"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_length=120,
    num_return_sequences=3,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    do_sample=True
)

for i, story in enumerate(output):
    print(f"Story {i+1}:\n")
    print(tokenizer.decode(story, skip_special_tokens=True))
    print("\n" + "="*80 + "\n")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Story 1:

Once upon a time in a forgotten kingdom, a glowing bird led a lost child to a hidden village made of crystal. Inside a hidden village, a hidden kingdom rested hidden away a forgotten child. When a warrior discovered a way to unlock the kingdom's secrets, they were able to change the weather. When a child discovered that anything she ever knew, she could breathe fire. Soon, her village changed the weather. When a child named Ibis learned to sing, they sang songs that changed the weather. When a boy named Kairu learned to dance, his songs changed the weather. When a


Story 2:

Once upon a time in a forgotten kingdom, a glowing bird led a lost child to a hidden village made of crystal. The child discovered it could speak to a place far, far before sunrise.


Story 3:

Once upon a time in a forgotten kingdom, a glowing bird led a lost child to a hidden village made of crystal. When they arrived at a hidden village made of crystal, they were able to learn about the past.

Learn t